In [1]:
import random
from rdkit import Chem
from molpher.core import MolpherMol, MolpherAtom
from molpher.core.morphing.operators import MorphingOperator
from rdkit.Chem.EnumerateStereoisomers import EnumerateStereoisomers, StereoEnumerationOptions
from rdkit.Chem import rdChemReactions
from rdkit.Chem import rdmolops
from rdkit.Chem import Descriptors  
from molpher.core import ExplorationTree as ETree

class AminoAcidConjugation(MorphingOperator):
    def __init__(self):
        super(AminoAcidConjugation, self).__init__()
        self._name = "Amino Acid Conjugation (Gly/Tau/Gln)"
        self._target_atoms = []
        self.PATTERN = Chem.MolFromSmarts("[CX3](=O)[OX2H]") 
        # Σε όλα, το Άζωτο (Ν) που θα επιτεθεί είναι το ΠΡΩΤΟ άτομο (index 0)
        self.TEMPLATES = {
            "Glycine": Chem.MolFromSmiles("NCC(=O)O"),
            "Taurine": Chem.MolFromSmiles("NCCS(=O)(=O)O"),
            "Glutamine": Chem.MolFromSmiles("N[C@@H](CCC(=O)N)C(=O)O") # Διατήρηση στερεοχημείας
        }

    def setOriginal(self, mol):
        super(AminoAcidConjugation, self).setOriginal(mol)
        self._target_atoms = []
        if not self.original: return
        rdkit_mol = self.original.asRDMol()
        if rdkit_mol is None: return
        
        matches = rdkit_mol.GetSubstructMatches(self.PATTERN)
        for match in matches:
            # Κρατάμε το index του Καρβονυλικού Άνθρακα (index 0) και του -OH (index 2)
            self._target_atoms.append((match[0], match[2]))

    def morph(self):
        if not self.original or not self._target_atoms:
            return MolpherMol(other=self.original.asRDMol())
        
        rdkit_mol = self.original.asRDMol()
        try:
            chosen_c_idx, chosen_oh_idx = random.choice(self._target_atoms)
            template_name = random.choice(list(self.TEMPLATES.keys()))
            template_mol = self.TEMPLATES[template_name]
            combined = Chem.CombineMols(rdkit_mol, template_mol)
            rw_combined = Chem.RWMol(combined)
            n_amino_idx = rdkit_mol.GetNumAtoms()
            rw_combined.AddBond(chosen_c_idx, n_amino_idx, Chem.BondType.SINGLE)
            rw_combined.RemoveAtom(chosen_oh_idx)
            
            new_mol = rw_combined.GetMol()
            
            # Λόγω του RemoveAtom, ο index του n_amino_idx μετατοπίστηκε κατά -1 
            # αν ο chosen_oh_idx ήταν μικρότερος, αλλά για ασφάλεια κάνουμε reset στο στοχευμένο καρβονύλιο 
            # και στο άζωτο σαρώνοντας το γράφημα.
            for atom in new_mol.GetAtoms():
                if atom.GetAtomicNum() in [6, 7]: # Άνθρακας καρβονυλίου και Άζωτο αμιδίου
                    atom.SetNoImplicit(False)
                    atom.SetNumExplicitHs(0)
                    atom.SetFormalCharge(0)
            
            new_mol.UpdatePropertyCache(strict=False)
            Chem.SanitizeMol(new_mol, Chem.SanitizeFlags.SANITIZE_ALL)
            Chem.AssignStereochemistry(new_mol, cleanIt=True, force=True)
            
            return MolpherMol(other=new_mol)
            
        except:
            return MolpherMol(other=rdkit_mol)

    def getName(self): return self._name


amino_op = AminoAcidConjugation()

class FindClosest:
    def __init__(self):
        self.closest_mol = None
        self.closest_distance = None
    def __call__(self, morph):
        if not self.closest_mol or self.closest_distance > morph.dist_to_target:
            self.closest_mol = morph
            self.closest_distance = morph.dist_to_target


start_mol = MolpherMol("O=C(O)c1ccccc1") # Βενζοϊκό Οξύ
target_mol = MolpherMol("O=C(NCC(=O)O)c1ccccc1") # Ιππουρικό Οξύ (Σύζευξη με Γλυκίνη)
tree = ETree.create(source=start_mol, target=target_mol)
tree.morphing_operators = (amino_op,)

closest_info = FindClosest()
print("--- STARTING AMINO ACID CONJUGATION TREE SEARCH ---")
while not tree.path_found:
    tree.generateMorphs()
    tree.sortMorphs()
    tree.filterMorphs()
    tree.extend()
    tree.prune()
    tree.traverse(closest_info)
        
    print(f"Generation #{tree.generation_count}")
    print(f"Molecules in tree: {tree.mol_count}")
    print(f"Closest to target: {closest_info.closest_mol.getSMILES()} (Distance: {closest_info.closest_distance:.4f})")
    print("-" * 40)
        
    if tree.path_found or tree.generation_count >= 5:
        break

print("\nSearch finished!")
if tree.path_found:
    print("SUCCESS: Target reached! The tree found the exact Amino Acid Conjugate.")
    
    # =========================================================================
    # ΕΛΕΓΧΟΣ FALSE POSITIVE ΠΑΓΙΔΩΝ
    # =========================================================================
print("\n=== RUNNING FALSE POSITIVE TRAP TESTS ===")
traps = {
        "Ester Trap (Δεν έχει ελεύθερο -OH)": "CC(=O)OCC",
        "Alcohol Trap (Φαινόλη - Όχι καρβοξύλιο)": "Oc1ccccc1",
        "Amide Trap (Αμίδιο - Όχι οξύ)": "CC(=O)NC"
}
    
for name, smiles in traps.items():
    mol = MolpherMol(smiles)
    amino_op.setOriginal(mol)
    product = amino_op.morph()
        
    safe = (product.getSMILES() == mol.getSMILES())
    print(f"{name}:")
    print(f"  SOURCE: {mol.getSMILES()}")
    print(f"  STATUS: {'SAFE (Passed)' if safe else 'VULNERABLE (Failed)'}")
    print("-" * 50)

--- STARTING AMINO ACID CONJUGATION TREE SEARCH ---
Generation #1
Molecules in tree: 4
Closest to target: O=C(O)CNC(=O)C1=CC=CC=C1 (Distance: 0.0000)
----------------------------------------

Search finished!
SUCCESS: Target reached! The tree found the exact Amino Acid Conjugate.

=== RUNNING FALSE POSITIVE TRAP TESTS ===
Ester Trap (Δεν έχει ελεύθερο -OH):
  SOURCE: CCOC(C)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
Alcohol Trap (Φαινόλη - Όχι καρβοξύλιο):
  SOURCE: OC1=CC=CC=C1
  STATUS: SAFE (Passed)
--------------------------------------------------
Amide Trap (Αμίδιο - Όχι οξύ):
  SOURCE: CNC(C)=O
  STATUS: SAFE (Passed)
--------------------------------------------------
